<a href="https://colab.research.google.com/github/vardhan23v/agentic-ai/blob/main/VALLUR_SREE_VARDHAN_NNM24CS285_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Title + overview (STUDENT: VALLUR SREE VARDHAN, NNM24CS285)
# Smart Campus Hackathon Assistant — CrewAI + Nexus
# Scenario: College Hackathon Coordination (24-hour inter-college build sprint).
# Why: hackathons need parallel tracks (hacking, mentoring, judging, food, power,
# networking) in one venue — ideal for planner + ops + risk + comms split.

This notebook builds a 4-agent CrewAI crew where every LLM call uses the Nexus API.
Deterministic Python does the maths; Open-Meteo gives real sky data; Nexus drafts notices.

## What you will learn
- How a natural-language request flows through 4 chained tasks
- Why each agent has different tools (no overlap)
- How facts, assumptions, and [TBD] placeholders stay separated


# 02 — Problem statement and objectives
## Problem
Hackathon planning lives in scattered chats: themes, team sizes, judging rules,
mentor slots, power/internet, food, prizes and safety cover get decided late.
A single chatbot answer mixes guesses with facts and gives no costed timetable.

## Objectives
1. Accept one natural-language hackathon request.
2. Parse it into confirmed facts vs missing items (never invent dates/venues/guests).
3. Produce a validated timetable, cost estimate, squad/resource map (Python math).
4. Ground outdoor-risk notes in REAL Open-Meteo data (or say unavailable).
5. Score risks with fixed rules + mitigations.
6. Draft a student notice with the Nexus LLM and assemble ONE final answer.


# 03 — Architecture and workflow
## Pipeline
User text -> check_input() -> Brief parse (Tool 1)
-> Timetable (Tool 2) -> Cost (Tool 3) + Squad resources (Tool 4)
-> SkyWatch weather (Tool 5, live) -> Hazard score (Tool 6)
-> Notice draft via Nexus (Tool 7) -> ResponseAssembler (Tool 8)
-> ONE final response + open questions.

## Crew chaining (CrewAI context=)
- Task 1 (Design Lead): brief + draft timetable skeleton.
- Task 2 (Ops & Finance, context=[T1]): timetable + budget + resources.
- Task 3 (Safety & Insight, context=[T1, T2]): weather + risk register.
- Task 4 (Outreach, context=[T1, T2, T3]): notice + final assembled answer.

## Fact discipline
Tools return JSON with facts vs assumptions. Agents must copy [TBD] forward,
never fill it silently. Weather labelled live / unavailable / simulated.


In [2]:
# 04 — Dependency installation
# crewai[tools]: the ONLY agent framework (Agent, Task, Crew, Process, BaseTool).
# requests: HTTP for Open-Meteo. openai: OpenAI-compatible client pointed at Nexus.
# pydantic: input schemas for the 8 tools.
%pip install -q "crewai[tools]" requests openai pydantic


In [3]:
# 05 — Imports
# Standard: json, re, os, textwrap, datetime. HTTP: requests.
# Colab secrets: google.colab.userdata (guarded so local runs do not crash).
# CrewAI: LLM, Agent, Task, Crew, Process + BaseTool. Pydantic: BaseModel, Field.
import json, re, os, textwrap
from datetime import datetime, timedelta
import requests
try:
    from google.colab import userdata
except Exception:
    userdata = None
import openai
from crewai import LLM, Agent, Task, Crew, Process
try:
    from crewai.tools import BaseTool
except ImportError:
    from crewai_tools import BaseTool
from pydantic import BaseModel, Field
print("imports ok")


imports ok


# 06 — Nexus API configuration
# Nexus is OpenAI-compatible. Course demo (Text_to_Text.ipynb) uses:
# base_url=https://nexusapi.navigatelabs.ai, model=gpt-4.1-nano, Bearer key.
# This notebook reads secrets NEXUSAI_API_KEY / NEXUSAI_BASE_URL / NEXUSAI_MODEL_ID,
# builds ONE shared CrewAI LLM, and exposes nexus_client() for the notice tool.
# No other LLM provider is used anywhere. Keys are never printed.


In [4]:
# Nexus config — secrets stay in Colab Secrets panel
def _secret(name):
    if userdata is not None:
        try:
            v = userdata.get(name)
            if v:
                return v
        except Exception:
            pass
    return os.environ.get(name)
NEXUSAI_API_KEY = _secret("NEXUSAI_API_KEY") or _secret("api_key")
NEXUSAI_BASE_URL = _secret("NEXUSAI_BASE_URL") or "https://nexusapi.navigatelabs.ai"
NEXUSAI_MODEL_ID = _secret("NEXUSAI_MODEL_ID") or "gpt-4.1-nano"
print(f"base={NEXUSAI_BASE_URL} model={NEXUSAI_MODEL_ID}")
if not NEXUSAI_API_KEY:
    print("MISSING: add Colab secret NEXUSAI_API_KEY (or api_key), then re-run.")
    NEXUSAI_OK, nexus_llm = False, None
else:
    print(f"key present (len {len(NEXUSAI_API_KEY)}; value hidden).")
    try:
        nexus_llm = LLM(model=NEXUSAI_MODEL_ID, base_url=NEXUSAI_BASE_URL, api_key=NEXUSAI_API_KEY, temperature=0.6)
        def nexus_client():
            return openai.OpenAI(api_key=NEXUSAI_API_KEY, base_url=NEXUSAI_BASE_URL)
        NEXUSAI_OK = True
        print("shared Nexus LLM built.")
    except Exception as e:
        print("config error:", type(e).__name__, str(e)[:200])
        NEXUSAI_OK, nexus_llm = False, None


base=https://nexusapi.navigatelabs.ai model=gpt-4.1-nano
key present (len 25; value hidden).
shared Nexus LLM built.


# 07 — Nexus connection test
# One tiny chat call proves the key + endpoint + model work.
# On failure we set NEXUSAI_OK=False and refuse other providers.


In [5]:
# Minimal Nexus test — max 10 tokens, no fallback
if not NEXUSAI_OK:
    print("SKIP: Nexus not configured; crew cells will wait for the key.")
else:
    try:
        r = nexus_client().chat.completions.create(model=NEXUSAI_MODEL_ID, max_tokens=10, messages=[{"role": "user", "content": "Reply with exactly: NEXUS-OK"}])
        print("Nexus reply:", r.choices[0].message.content.strip())
        print("Nexus connection: OK")
    except Exception as e:
        print("Nexus connection FAILED:", type(e).__name__, str(e)[:300])
        print("Fix the secret/network and re-run. No other provider will be used.")
        NEXUSAI_OK = False


Nexus reply: NEXUS-OK
Nexus connection: OK


# 08 — External API research and independent test
# Choice: Open-Meteo Forecast API (https://open-meteo.com).
# Why: free, no key, HTTPS GET returning JSON — perfect for a student notebook
# and for outdoor hackathon segments (opening, food stalls, demo expo).
# Docs: https://open-meteo.com/en/docs — params used: latitude, longitude,
# hourly=temperature_2m, daily=temperature_2m_max, current=temperature_2m.
# Below is a REAL request for NITTE (Mangalore ~12.97N, 74.88E) BEFORE tools use it.


In [6]:
# Live Open-Meteo probe for NITTE Mangalore (independent of tools)
WMO = {0: "Clear", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Icing fog", 51: "Light drizzle", 61: "Rain", 71: "Snow", 80: "Showers", 95: "Storm"}
try:
    rr = requests.get("https://api.open-meteo.com/v1/forecast", params={"latitude": 12.97, "longitude": 74.88, "current": "temperature_2m,weather_code", "daily": "temperature_2m_max,precipitation_probability_max", "timezone": "Asia/Kolkata", "forecast_days": 2}, timeout=15)
    print("HTTP:", rr.status_code)
    if rr.status_code == 200:
        dd = rr.json()
        print("resolved:", dd.get("latitude"), dd.get("longitude"), "| tz:", dd.get("timezone"))
        print("now:", dd["current"].get("temperature_2m"), "C |", WMO.get(dd["current"].get("weather_code"), "n/a"))
        print("day0 max:", dd["daily"]["temperature_2m_max"][0], "C | rain%:", dd["daily"]["precipitation_probability_max"][0])
        print("LIVE data confirmed — tools may reuse this endpoint.")
    else:
        print("API unavailable (HTTP", rr.status_code, ") — tools must report unavailable.")
except Exception as e:
    print("probe failed:", type(e).__name__, str(e)[:200], "— tools must report unavailable.")


HTTP: 200
resolved: 12.970123 74.86363 | tz: Asia/Kolkata
now: 29.1 C | Light drizzle
day0 max: 29.5 C | rain%: 100
LIVE data confirmed — tools may reuse this endpoint.


# 09 — Custom tool implementation (8 tools, all BaseTool + Pydantic)
# T1 HackathonBriefParser: facts vs missing. T2 TimetableArchitect: no-overlap slots.
# T3 CostEstimator: INR math + buffer. T4 SquadAllocator: staff ratios + kit list.
# T5 SkyWatch: REAL Open-Meteo fetch. T6 HazardRadar: fixed risk rules.
# T7 NoticeDrafter: Nexus LLM only. T8 ResponseAssembler: one final text.
# A shared CALL_LOG records every invocation as test evidence.


In [7]:
# Shared call log + TOOL 1: HackathonBriefParser (deterministic)
CALL_LOG = []
def _log(tool, args, ok=True, note=""):
    CALL_LOG.append({"tool": tool, "args": str(args)[:250], "ok": ok, "note": note})
class BriefIn(BaseModel):
    user_request: str = Field(..., description="Raw hackathon request text.")
class HackathonBriefParser(BaseTool):
    name: str = "HackathonBriefParser"
    description: str = "Parse a hackathon request into confirmed facts, missing items and questions. Never invents dates or venues."
    args_schema: type[BaseModel] = BriefIn
    def _run(self, user_request: str) -> str:
        try:
            if not user_request or not user_request.strip():
                _log(self.name, "empty", False, "empty")
                return json.dumps({"error": "Empty request."})
            t = user_request.lower()
            if any(k in t for k in ["hackathon", "hack", "codefest", "build-a-thon"]):
                et = "hackathon"
            elif "guest" in t and ("speaker" in t or "visit" in t):
                et = "guest visit"
            elif "coding competition" in t or "contest" in t:
                et = "coding competition"
            else:
                et = "campus event"
            m = re.search(r"(\d{2,4})\s*(students|participants|hackers|attendees|members|people)", t)
            head = int(m.group(1)) if m else None
            h = re.search(r"(\d+)\s*[- ]?hour", t)
            hours = int(h.group(1)) if h else (24 if "24" in t else None)
            venue = "NITTE campus" if "nitte" in t else (None if "venue" not in t and "campus" not in t else "campus venue (name TBD)")
            miss = [k for k, v in [("headcount", head), ("duration_hours", hours), ("venue", venue)] if not v]
            facts = {"event_type": et}
            if head: facts["headcount"] = head
            if hours: facts["duration_hours"] = hours
            if venue: facts["venue_hint"] = venue
            _log(self.name, user_request[:60], True, et)
            return json.dumps({"facts_from_user": facts, "missing": miss or ["none critical"], "questions": [f"Please confirm {x}?" for x in miss][:4], "rule": "Unlisted items are NOT confirmed."}, indent=2)
        except Exception as e:
            _log(self.name, "err", False, str(e)[:80])
            return json.dumps({"error": f"{type(e).__name__}: {e}"})
print("Tool 1 ready: HackathonBriefParser")


Tool 1 ready: HackathonBriefParser


In [8]:
# TOOL 2: TimetableArchitect — equal sequential slots, overlap-proof by construction
class TableIn(BaseModel):
    duration_hours: float = Field(..., description="Total hours, 1..48.")
    agenda: str = Field(..., description="Comma-separated items, e.g. check-in, hacking, judging.")
    start: str = Field("09:00", description="HH:MM 24h.")
class TimetableArchitect(BaseTool):
    name: str = "TimetableArchitect"
    description: str = "Build a sequential hackathon timetable with equal slots; rejects overfull agendas."
    args_schema: type[BaseModel] = TableIn
    def _run(self, duration_hours: float, agenda: str, start: str = "09:00") -> str:
        try:
            items = [a.strip() for a in agenda.split(",") if a.strip()]
            if not items:
                return json.dumps({"error": "No agenda items."})
            if not (1 <= duration_hours <= 48):
                return json.dumps({"error": "duration_hours must be 1..48."})
            try:
                base = datetime.strptime(start, "%H:%M")
            except ValueError:
                return json.dumps({"error": "start must be HH:MM."})
            slot = (duration_hours * 60) / len(items)
            if slot < 15:
                return json.dumps({"error": f"slot {slot:.0f} min < 15 min minimum; cut items or add hours."})
            cur, rows = base, []
            for a in items:
                end = cur + timedelta(minutes=slot)
                rows.append({"item": a, "start": cur.strftime("%H:%M"), "end": end.strftime("%H:%M")})
                cur = end
            _log(self.name, f"{duration_hours}h/{len(items)}", True, "sequential ok")
            return json.dumps({"start": start, "total_hours": duration_hours, "slot_min": round(slot), "rows": rows, "check": "sequential, no overlaps"}, indent=2)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})
print("Tool 2 ready: TimetableArchitect")


Tool 2 ready: TimetableArchitect


In [9]:
# TOOL 3: CostEstimator — INR arithmetic with meal tiers + buffer
class CostIn(BaseModel):
    headcount: int = Field(..., description="People, 1..5000.")
    duration_hours: float = Field(..., description="Hours, 1..48.")
    food_pp: float = Field(300.0, description="INR per meal per person.")
    kit_pp: float = Field(150.0, description="INR kit per person.")
    fixed: float = Field(25000.0, description="INR fixed (stage/power/prizes).")
    buffer_pct: float = Field(10.0, description="Buffer %.")
class CostEstimator(BaseTool):
    name: str = "CostEstimator"
    description: str = "Estimate hackathon cost in INR: food tiers + kits + fixed + buffer. Estimates only."
    args_schema: type[BaseModel] = CostIn
    def _run(self, headcount: int, duration_hours: float, food_pp: float = 300.0, kit_pp: float = 150.0, fixed: float = 25000.0, buffer_pct: float = 10.0) -> str:
        try:
            if not (1 <= headcount <= 5000):
                return json.dumps({"error": "headcount must be 1..5000."})
            if not (1 <= duration_hours <= 48):
                return json.dumps({"error": "duration_hours must be 1..48."})
            meals = 1 if duration_hours <= 8 else (2 if duration_hours <= 16 else 3)
            food = headcount * food_pp * meals
            kits = headcount * kit_pp
            sub = food + kits + fixed
            buf = sub * buffer_pct / 100.0
            _log(self.name, f"{headcount}p/{duration_hours}h", True, "estimate")
            return json.dumps({"inputs": {"headcount": headcount, "hours": duration_hours, "meals": meals}, "lines_inr": {"food": round(food, 2), "kits": round(kits, 2), "fixed": fixed, "subtotal": round(sub, 2), "buffer": round(buf, 2), "total": round(sub + buf, 2)}, "label": "ESTIMATE, not a vendor quote."}, indent=2)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})
print("Tool 3 ready: CostEstimator")


Tool 3 ready: CostEstimator


In [10]:
# TOOL 4: SquadAllocator — volunteers, mentors, judges from fixed ratios + kit checklist
class SquadIn(BaseModel):
    headcount: int = Field(..., description="People, 1..5000.")
    outdoor: bool = Field(False, description="Any outdoor segment?")
class SquadAllocator(BaseTool):
    name: str = "SquadAllocator"
    description: str = "Derive volunteer/mentor/judge counts and kit list from headcount using fixed ratios."
    args_schema: type[BaseModel] = SquadIn
    def _run(self, headcount: int, outdoor: bool = False) -> str:
        try:
            if not (1 <= headcount <= 5000):
                return json.dumps({"error": "headcount must be 1..5000."})
            vols = max(6, -(-headcount // 15))
            mentors = max(3, -(-headcount // 40))
            judges = 3 if headcount <= 100 else (5 if headcount <= 250 else 7)
            kit = ["stage + PA", "projector/LED wall", "mics", "Wi-Fi + power strips", "registration desks", "first-aid kit", "drinking water", "id bands"]
            if outdoor:
                kit += ["tents/shade", "portable fans", "rain cover"]
            _log(self.name, f"{headcount}p", True, f"{vols}v/{mentors}m/{judges}j")
            return json.dumps({"crew": {"volunteers": vols, "mentors": mentors, "judges": judges, "ratio_note": "1:15 vols, 1:40 mentors, tiered judges"}, "kit": kit, "still_needed": ["venue booking ref", "prize sponsor list", "judge names"]}, indent=2)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})
print("Tool 4 ready: SquadAllocator")


Tool 4 ready: SquadAllocator


In [11]:
# TOOL 5: SkyWatch — REAL Open-Meteo fetch; honest unavailable on any failure
class SkyIn(BaseModel):
    lat: float = Field(..., description="Latitude -90..90.")
    lon: float = Field(..., description="Longitude -180..180.")
    days: int = Field(2, description="1..7 forecast days.")
class SkyWatch(BaseTool):
    name: str = "SkyWatch"
    description: str = "Fetch real forecast from Open-Meteo; returns live data or unavailable — never invents weather."
    args_schema: type[BaseModel] = SkyIn
    def _run(self, lat: float, lon: float, days: int = 2) -> str:
        try:
            if not (-90 <= lat <= 90 and -180 <= lon <= 180):
                return json.dumps({"status": "unavailable", "reason": "bad coordinates"})
            days = max(1, min(7, int(days)))
            r = requests.get("https://api.open-meteo.com/v1/forecast", params={"latitude": lat, "longitude": lon, "current": "temperature_2m,weather_code", "daily": "temperature_2m_max,precipitation_probability_max", "timezone": "Asia/Kolkata", "forecast_days": days}, timeout=15)
            if r.status_code != 200:
                _log(self.name, f"{lat},{lon}", False, f"HTTP {r.status_code}")
                return json.dumps({"status": "unavailable", "reason": f"HTTP {r.status_code}", "guidance": "Move outdoor items indoors; no forecast invented."})
            d = r.json()
            _log(self.name, f"{lat},{lon}", True, "live")
            return json.dumps({"status": "live", "source": "Open-Meteo (real API data)", "place": {"asked": [lat, lon], "fixed": [d.get("latitude"), d.get("longitude")], "tz": d.get("timezone")}, "now": {"temp_c": d["current"].get("temperature_2m"), "code": d["current"].get("weather_code")}, "days": [{"date": d["daily"]["time"][i], "max_c": d["daily"]["temperature_2m_max"][i], "rain_pct": d["daily"]["precipitation_probability_max"][i]} for i in range(len(d["daily"]["time"]))], "tip": "Outdoor cover if any rain_pct >= 50 or max_c >= 36."}, indent=2)
        except requests.exceptions.Timeout:
            _log(self.name, "timeout", False, "timeout")
            return json.dumps({"status": "unavailable", "reason": "timeout", "guidance": "Proceed with indoor backup."})
        except Exception as e:
            _log(self.name, "err", False, str(e)[:80])
            return json.dumps({"status": "unavailable", "reason": f"{type(e).__name__}: {str(e)[:150]}"})
print("Tool 5 ready: SkyWatch")


Tool 5 ready: SkyWatch


In [12]:
# TOOL 6: HazardRadar — fixed transparent rules, no LLM judgement
class RiskIn(BaseModel):
    outdoor: bool = Field(False, description="Outdoor segment?")
    headcount: int = Field(100, description="People.")
    sky: str = Field("unavailable", description="SkyWatch JSON or 'unavailable'.")
class HazardRadar(BaseTool):
    name: str = "HazardRadar"
    description: str = "Score heat/rain/crowd/power/judging risks with fixed thresholds and mitigations."
    args_schema: type[BaseModel] = RiskIn
    def _run(self, outdoor: bool = False, headcount: int = 100, sky: str = "unavailable") -> str:
        try:
            risks = []
            try:
                s = json.loads(sky) if isinstance(sky, str) and sky.strip().startswith("{") else {}
            except Exception:
                s = {}
            live = s.get("status") == "live"
            worst_rain, worst_heat = 0, 0
            for dd in s.get("days", []) if live else []:
                worst_rain = max(worst_rain, dd.get("rain_pct") or 0)
                worst_heat = max(worst_heat, dd.get("max_c") or 0)
            if outdoor and live and worst_rain >= 50:
                risks.append({"risk": "rain on outdoor segment", "level": "HIGH", "fix": "tent + indoor backup hall"})
            elif outdoor and not live:
                risks.append({"risk": "unknown sky for outdoor segment", "level": "MEDIUM", "fix": "book indoor backup; recheck sky 24h before"})
            if live and worst_heat >= 36:
                risks.append({"risk": "heat stress", "level": "HIGH", "fix": "shade, water stations, shift outdoor slots to morning"})
            if headcount >= 250:
                risks.append({"risk": "crowd + queues", "level": "MEDIUM", "fix": "two registration desks, staggered lunch"})
            risks += [{"risk": "power/Wi-Fi drop mid-hacking", "level": "MEDIUM", "fix": "UPS per row, backup router + offline judging sheets"}, {"risk": "judging overrun", "level": "LOW", "fix": "3-min demo timer + overflow room"}]
            _log(self.name, f"out={outdoor}/{headcount}", True, f"{len(risks)} risks")
            return json.dumps({"sky_used": "live" if live else "unavailable (assumptions flagged)", "risks": risks}, indent=2)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})
print("Tool 6 ready: HazardRadar")


Tool 6 ready: HazardRadar


In [13]:
# TOOL 7: NoticeDrafter — the ONLY LLM-powered tool; Nexus exclusively
class NoticeIn(BaseModel):
    title: str = Field(..., description="Event title.")
    facts: str = Field(..., description="Confirmed facts only.")
    audience: str = Field("students", description="students|faculty|guests|all.")
class NoticeDrafter(BaseTool):
    name: str = "NoticeDrafter"
    description: str = "Draft a campus notice from given facts using the Nexus LLM. Unknowns become [TBD]; never invents links or dates."
    args_schema: type[BaseModel] = NoticeIn
    def _run(self, title: str, facts: str, audience: str = "students") -> str:
        try:
            if not NEXUSAI_OK:
                _log(self.name, title, False, "no nexus")
                return json.dumps({"status": "skipped", "reason": "Nexus not configured; no other provider used.", "template": f"{title} | {facts} | Date/Venue/Link: [TBD]"})
            msg = (f"Write a short campus notice for {audience}.\nTitle: {title}\nConfirmed facts: {facts}\nRules: use ONLY these facts; unknowns as [TBD]; no invented dates, links, contacts, or bookings; under 160 words.")
            r = nexus_client().chat.completions.create(model=NEXUSAI_MODEL_ID, max_tokens=350, messages=[{"role": "user", "content": msg}])
            _log(self.name, title, True, "nexus draft")
            return json.dumps({"status": "drafted-by-nexus", "notice": r.choices[0].message.content.strip()}, indent=2)
        except Exception as e:
            _log(self.name, "err", False, str(e)[:80])
            return json.dumps({"status": "failed", "reason": f"{type(e).__name__}: {str(e)[:200]}", "guidance": "Crew continues; notice marked pending."})
print("Tool 7 ready: NoticeDrafter (Nexus-only)")


Tool 7 ready: NoticeDrafter (Nexus-only)


In [14]:
# TOOL 8: ResponseAssembler — deterministic merge into ONE readable answer
class PackIn(BaseModel):
    title: str = Field(..., description="Response title.")
    parts_json: str = Field(..., description="JSON object of section->text.")
class ResponseAssembler(BaseTool):
    name: str = "ResponseAssembler"
    description: str = "Merge prior outputs into one final response; preserves [TBD] and never claims bookings done."
    args_schema: type[BaseModel] = PackIn
    def _run(self, title: str, parts_json: str) -> str:
        try:
            parts = json.loads(parts_json)
            if not isinstance(parts, dict) or not parts:
                return json.dumps({"error": "parts_json must be a non-empty object."})
            out = [title.upper(), "=" * len(title)]
            for k, v in parts.items():
                out += ["", f"## {k}", str(v)[:1500]]
            out += ["", "## Facts vs assumptions", "Facts: only what the user gave + live SkyWatch data. Everything else is an assumption or [TBD].", "No venue, booking, budget, link, or guest is confirmed unless the organisers confirm it."]
            _log(self.name, title, True, f"{len(parts)} sections")
            return "\n".join(out)
        except Exception as e:
            return json.dumps({"error": f"{type(e).__name__}: {e}"})
print("Tool 8 ready: ResponseAssembler")


Tool 8 ready: ResponseAssembler


In [15]:
# Demo: run all 8 tools once (definition alone does not count as execution)
d1, d2, d3, d4 = HackathonBriefParser(), TimetableArchitect(), CostEstimator(), SquadAllocator()
d5, d6, d7, d8 = SkyWatch(), HazardRadar(), NoticeDrafter(), ResponseAssembler()
print("== D1 brief =="); print(d1._run("Plan a 24-hour inter-college hackathon at NITTE for 200 students."))
print("\n== D2 timetable =="); print(d2._run(8.0, "Check-in, Opening, Hacking block 1, Lunch, Hacking block 2, Judging, Prizes", "09:00"))
print("\n== D3 cost =="); print(d3._run(200, 24.0))
print("\n== D4 squad =="); print(d4._run(200, True))
print("\n== D5 sky (live NITTE) ==")
sky_demo = d5._run(12.97, 74.88, 2); print(sky_demo[:900])
print("\n== D6 hazards =="); print(d6._run(True, 200, sky_demo))
print("\n== D7 notice ==")
print(d7._run("NITTE HackNight 2026", "24-hour hackathon, ~200 students, NITTE campus. Date/venue/time: [TBD].", "students")[:700])
print("\n== D8 assemble ==")
print(d8._run("Demo Pack", json.dumps({"Plan": "HackNight draft", "Cost": "estimate only", "Next": "Confirm date + hall"}))[:500])
print("\nCALL_LOG entries:", len(CALL_LOG))


== D1 brief ==
{
  "facts_from_user": {
    "event_type": "hackathon",
    "headcount": 200,
    "duration_hours": 24,
    "venue_hint": "NITTE campus"
  },
  "missing": [
    "none critical"
  ],
  "questions": [],
  "rule": "Unlisted items are NOT confirmed."
}

== D2 timetable ==
{
  "start": "09:00",
  "total_hours": 8.0,
  "slot_min": 69,
  "rows": [
    {
      "item": "Check-in",
      "start": "09:00",
      "end": "10:08"
    },
    {
      "item": "Opening",
      "start": "10:08",
      "end": "11:17"
    },
    {
      "item": "Hacking block 1",
      "start": "11:17",
      "end": "12:25"
    },
    {
      "item": "Lunch",
      "start": "12:25",
      "end": "13:34"
    },
    {
      "item": "Hacking block 2",
      "start": "13:34",
      "end": "14:42"
    },
    {
      "item": "Judging",
      "start": "14:42",
      "end": "15:51"
    },
    {
      "item": "Prizes",
      "start": "15:51",
      "end": "16:59"
    }
  ],
  "check": "sequential, no overlaps"
}

== 

# 10 — Agent creation (4 specialists, distinct roles/tools, shared Nexus LLM)
# A1 Hackathon Design Lead: brief + timetable skeleton (Tools 1-2).
# A2 Ops & Finance Manager: timetable + cost + squad (Tools 2-4).
# A3 Safety & Insight Officer: sky + risk register (Tools 5-6).
# A4 Outreach Coordinator: notice + final pack (Tools 7-8).
# No two agents share the same job; all reason via nexus_llm.


In [16]:
# Instantiate tools once, then build the 4 agents
p_tool, t_tool = HackathonBriefParser(), TimetableArchitect()
c_tool, s_tool = CostEstimator(), SquadAllocator()
w_tool, r_tool = SkyWatch(), HazardRadar()
n_tool, f_tool = NoticeDrafter(), ResponseAssembler()
def build_crew_agents():
    if not NEXUSAI_OK or nexus_llm is None:
        raise RuntimeError("Nexus LLM missing — add NEXUSAI_API_KEY and re-run. No fallback provider.")
    a1 = Agent(role="Hackathon Design Lead", goal="Turn a raw request into confirmed facts, open questions and a workable agenda.", backstory="NITTE senior who has run inter-college build sprints; strict about facts vs guesses, never invents dates or halls.", tools=[p_tool, t_tool], llm=nexus_llm, verbose=True)
    a2 = Agent(role="Ops & Finance Manager", goal="Convert the agenda into a timed plan with INR estimates and crew/kit lists.", backstory="Lab operations volunteer who costs food, kits, power backup and volunteer shifts with Python calculators.", tools=[t_tool, c_tool, s_tool], llm=nexus_llm, verbose=True)
    a3 = Agent(role="Safety & Insight Officer", goal="Ground the plan in live sky data and score risks with backups.", backstory="Safety volunteer covering heat, rain, crowd, power and judging overruns; reports unavailable honestly.", tools=[w_tool, r_tool], llm=nexus_llm, verbose=True)
    a4 = Agent(role="Outreach Coordinator", goal="Write the student notice and assemble ONE final answer with next steps.", backstory="Club communications lead writing for students, faculty and guests; keeps [TBD] visible, claims nothing booked.", tools=[n_tool, f_tool], llm=nexus_llm, verbose=True)
    return a1, a2, a3, a4
try:
    ag1, ag2, ag3, ag4 = build_crew_agents()
    print("agents:", ag1.role, "|", ag2.role, "|", ag3.role, "|", ag4.role)
except RuntimeError as e:
    print("AGENTS WAITING:", e)
    ag1 = ag2 = ag3 = ag4 = None


agents: Hackathon Design Lead | Ops & Finance Manager | Safety & Insight Officer | Outreach Coordinator


# 11 — Task definitions and context dependencies
# T1 analyses the request (Tools 1-2). T2 costs/resources it (context=[T1]).
# T3 risks it with live sky (context=[T1, T2]). T4 notices + packs all
# (context=[T1, T2, T3]). Context= reuses facts instead of re-inventing them.


In [17]:
# Build 4 chained tasks for a given request
def build_crew_tasks(a1, a2, a3, a4, ask):
    t1 = Task(description=f"Analyse this hackathon request and draft the agenda skeleton. Use HackathonBriefParser then TimetableArchitect. Facts vs [TBD] must stay labelled.\nREQUEST: {ask}\nNITTE sky reference: lat 12.97, lon 74.88 (Safety officer fetches it later).", expected_output="JSON brief (facts/missing/questions) + draft timetable rows.", agent=a1)
    t2 = Task(description="Using Task 1's brief + timetable, finalise the timetable (TimetableArchitect), estimate cost (CostEstimator), list crew/kit (SquadAllocator). All numbers via tools, labelled ESTIMATE.", expected_output="Final timetable, INR cost table, crew/kit list.", agent=a2, context=[t1])
    t3 = Task(description="Using Tasks 1-2, fetch REAL sky via SkyWatch (12.97, 74.88), then score risks via HazardRadar. If sky is unavailable say so; never invent weather.", expected_output="Sky summary (live/unavailable) + risk register with fixes.", agent=a3, context=[t1, t2])
    t4 = Task(description="Using Tasks 1-3, draft the student notice (NoticeDrafter, Nexus) then merge EVERYTHING with ResponseAssembler into ONE final answer with open questions.", expected_output="Final assembled response with notice, plan, cost, risks, next steps.", agent=a4, context=[t1, t2, t3])
    return [t1, t2, t3, t4]
print("task builder ready: build_crew_tasks(a1..a4, ask) -> [t1..t4]")


task builder ready: build_crew_tasks(a1..a4, ask) -> [t1..t4]


# 12 — User input validation
# check_input() gates the crew: empty/off-topic never run; incomplete runs as
# provisional with visible questions. Edit HACK_REQUEST to try your own case.


In [18]:
# Editable request + validator
HACK_REQUEST = """Plan a 24-hour inter-college hackathon at NITTE for about 200 students.
Include team check-in, opening ceremony, hacking blocks, mentor clinics,
food breaks, mid-way review, final demos, judging and prize distribution."""
def check_input(text):
    if not text or not text.strip():
        return {"status": "empty", "message": "Request is empty — describe the hackathon."}
    t = text.lower()
    keys = ["hackathon", "hack", "event", "fest", "competition", "contest", "workshop", "seminar", "visit", "announce", "campus", "college", "student", "nitte", "judging", "mentor"]
    if not any(k in t for k in keys):
        return {"status": "offtopic", "message": "Not a campus request. Try 'Plan a hackathon...'."}
    miss = []
    if not re.search(r"\d{2,4}\s*(students|participants|hackers|attendees)", t):
        miss.append("headcount")
    if not re.search(r"\d+\s*[- ]?hour|24|overnight|one-day|two-day", t):
        miss.append("duration")
    if "nitte" not in t and "venue" not in t and "campus" not in t:
        miss.append("venue")
    return {"status": "ok", "missing": miss, "questions": [f"Please confirm {m}?" for m in miss]}
print("REQUEST:"); print(textwrap.fill(HACK_REQUEST.strip(), 88))
print("\ncheck:", json.dumps(check_input(HACK_REQUEST), indent=2))


REQUEST:
Plan a 24-hour inter-college hackathon at NITTE for about 200 students. Include team
check-in, opening ceremony, hacking blocks, mentor clinics, food breaks, mid-way review,
final demos, judging and prize distribution.

check: {
  "status": "ok",
  "missing": [],
  "questions": []
}


# 13 — Crew execution via run_campus_assistant()
# One function: validate -> build agents/tasks -> sequential kickoff ->
# collect intermediate Task outputs + CALL_LOG -> bundle with final text.
# Errors are caught so the notebook never crashes mid-assignment.


In [19]:
# Reusable runner — the only entry point tests use
def run_campus_assistant(user_request, verbose=False):
    pack = {"input": user_request, "stages": {}, "tools": [], "final": "", "error": None}
    try:
        v = check_input(user_request)
        pack["stages"]["validation"] = v
        if v["status"] in ("empty", "offtopic"):
            pack["final"] = f"Cannot proceed: {v['message']}"
            return pack
        a1, a2, a3, a4 = build_crew_agents()
        chores = build_crew_tasks(a1, a2, a3, a4, user_request.strip())
        pack["stages"]["plan"] = [f"T{i+1}->{c.agent.role}: {c.description[:80]}..." for i, c in enumerate(chores)]
        crew = Crew(agents=[a1, a2, a3, a4], tasks=chores, process=Process.sequential, verbose=verbose)
        pack["final"] = str(crew.kickoff())
        mid = {}
        for i, c in enumerate(chores):
            try:
                o = getattr(c, "output", None)
                mid[f"task{i+1}_{c.agent.role}"] = str(o.raw)[:1200] if o is not None and hasattr(o, "raw") else (str(o)[:1200] if o is not None else "(no output)")
            except Exception as e:
                mid[f"task{i+1}"] = f"(read failed: {type(e).__name__})"
        pack["stages"]["intermediate"] = mid
        pack["tools"] = [{"tool": e["tool"], "ok": e["ok"], "note": e["note"]} for e in CALL_LOG]
        return pack
    except Exception as e:
        pack["error"] = f"{type(e).__name__}: {str(e)[:400]}"
        pack["final"] = f"Workflow stopped safely. Diagnostic: {pack['error']}"
        return pack
print("runner ready: run_campus_assistant(text)")


runner ready: run_campus_assistant(text)


In [20]:
# Execute the default hackathon request (needs Nexus key; else waits politely)
if NEXUSAI_OK and ag1 is not None:
    CALL_LOG.clear()
    demo = run_campus_assistant(HACK_REQUEST, verbose=False)
    print("validation:", json.dumps(demo["stages"].get("validation"), indent=2))
    print("\ntools fired:", json.dumps(demo["tools"], indent=2)[:800])
    print("\n--- FINAL (first 2500 chars) ---\n" + demo["final"][:2500])
else:
    print("WAITING: add NEXUSAI_API_KEY then re-run. Validator + tool demos above already prove logic.")


validation: {
  "status": "ok",
  "missing": [],
  "questions": []
}

tools fired: []

--- FINAL (first 2500 chars) ---
Workflow stopped safely. Diagnostic: RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.


# 14 — Three normal test cases
# T1: full hackathon (plan + cost + sky + risks + notice).
# T2: guest-visit variant (same crew, different brief — proves generality).
# T3: notice-only coding-competition request (placeholders + checklist).
# Each shows input -> tools fired -> final excerpt -> honest observation.
# Crew rows need Nexus; deterministic fallbacks always run.


In [21]:
# TEST 1: one-day student tech event inside a hackathon-style day
t1ask = "Plan a one-day student technology event at NITTE with workshops, coding competitions, lunch and a closing session for 150 students."
print("TEST 1 INPUT:", t1ask)
print("check:", json.dumps(check_input(t1ask), indent=2))
print("brief:", p_tool._run(t1ask)[:400])
print("slots:", t_tool._run(8.0, "Opening, Workshop, Coding contest, Lunch, Demos, Closing", "09:00")[:400])
print("cost:", c_tool._run(150, 8.0)[:300])
if NEXUSAI_OK and ag1 is not None:
    CALL_LOG.clear(); r1 = run_campus_assistant(t1ask)
    print("crew tools:", json.dumps(r1["tools"], indent=2)[:600])
    print("FINAL excerpt:\n" + r1["final"][:1800])
    print("\nOBSERVE: crew path exercised; verify timetable rows + cost lines match tool math.")
else:
    print("OBSERVE (no key): deterministic tools above prove the pipeline shape; crew excerpt pending key.")


TEST 1 INPUT: Plan a one-day student technology event at NITTE with workshops, coding competitions, lunch and a closing session for 150 students.
check: {
  "status": "ok",
  "missing": [],
  "questions": []
}
brief: {
  "facts_from_user": {
    "event_type": "coding competition",
    "headcount": 150,
    "venue_hint": "NITTE campus"
  },
  "missing": [
    "duration_hours"
  ],
  "questions": [
    "Please confirm duration_hours?"
  ],
  "rule": "Unlisted items are NOT confirmed."
}
slots: {
  "start": "09:00",
  "total_hours": 8.0,
  "slot_min": 80,
  "rows": [
    {
      "item": "Opening",
      "start": "09:00",
      "end": "10:20"
    },
    {
      "item": "Workshop",
      "start": "10:20",
      "end": "11:40"
    },
    {
      "item": "Coding contest",
      "start": "11:40",
      "end": "13:00"
    },
    {
      "item": "Lunch",
      "start": "13:00",
      "end": "14
cost: {
  "inputs": {
    "headcount": 150,
    "hours": 8.0,
    "meals": 1
  },
  "lines_inr": {
   

In [22]:
# TEST 2: guest speaker visit (welcome, lecture, interaction, lunch, send-off)
t2ask = "Plan a guest speaker campus visit at NITTE: welcome session, expert lecture, student interaction, lunch and departure for 80 attendees."
print("TEST 2 INPUT:", t2ask)
print("check:", json.dumps(check_input(t2ask), indent=2))
print("brief:", p_tool._run(t2ask)[:400])
print("slots:", t_tool._run(6.0, "Welcome, Lecture, Interaction, Lunch, Send-off", "10:00")[:400])
print("risks:", r_tool._run(False, 80, "unavailable")[:400])
if NEXUSAI_OK and ag1 is not None:
    CALL_LOG.clear(); r2 = run_campus_assistant(t2ask)
    print("crew tools:", json.dumps(r2["tools"], indent=2)[:600])
    print("FINAL excerpt:\n" + r2["final"][:1800])
    print("\nOBSERVE: guest identity/time must stay [TBD]; no booking claimed.")
else:
    print("OBSERVE (no key): brief correctly types this as guest visit; crew excerpt pending key.")


TEST 2 INPUT: Plan a guest speaker campus visit at NITTE: welcome session, expert lecture, student interaction, lunch and departure for 80 attendees.
check: {
  "status": "ok",
  "missing": [
    "duration"
  ],
  "questions": [
    "Please confirm duration?"
  ]
}
brief: {
  "facts_from_user": {
    "event_type": "guest visit",
    "headcount": 80,
    "venue_hint": "NITTE campus"
  },
  "missing": [
    "duration_hours"
  ],
  "questions": [
    "Please confirm duration_hours?"
  ],
  "rule": "Unlisted items are NOT confirmed."
}
slots: {
  "start": "10:00",
  "total_hours": 6.0,
  "slot_min": 72,
  "rows": [
    {
      "item": "Welcome",
      "start": "10:00",
      "end": "11:12"
    },
    {
      "item": "Lecture",
      "start": "11:12",
      "end": "12:24"
    },
    {
      "item": "Interaction",
      "start": "12:24",
      "end": "13:36"
    },
    {
      "item": "Lunch",
      "start": "13:36",
      "end": "14:48"
risks: {
  "sky_used": "unavailable (assumptions flagg

In [23]:
# TEST 3: announcement-only coding competition
t3ask = "Prepare an announcement for an upcoming campus coding competition for 120 students and list what organisers must still provide."
print("TEST 3 INPUT:", t3ask)
print("check:", json.dumps(check_input(t3ask), indent=2))
print("brief:", p_tool._run(t3ask)[:400])
print("notice:", n_tool._run("Campus Coding Competition", "Coding contest for ~120 students at NITTE. Date/venue/rules/link: [TBD].", "students")[:600])
print("OBSERVE: notice must contain [TBD] for date/venue/link; facts only from the request. LIMIT: without key, template path is used — label it as such.")


TEST 3 INPUT: Prepare an announcement for an upcoming campus coding competition for 120 students and list what organisers must still provide.
check: {
  "status": "ok",
  "missing": [
    "duration"
  ],
  "questions": [
    "Please confirm duration?"
  ]
}
brief: {
  "facts_from_user": {
    "event_type": "coding competition",
    "headcount": 120,
    "venue_hint": "campus venue (name TBD)"
  },
  "missing": [
    "duration_hours"
  ],
  "questions": [
    "Please confirm duration_hours?"
  ],
  "rule": "Unlisted items are NOT confirmed."
}
notice: {
  "status": "drafted-by-nexus",
  "notice": "**Campus Coding Competition**\n\nAttention NITTE students!  \nJoin our upcoming coding contest open to approximately 120 participants. Stay tuned for more details on the date, venue, and rules\u2014these will be announced soon.  \n\nPrepare to showcase your coding skills and compete with your peers. For updates, keep an eye on notice boards and the official campus portal.  \n\nGood luck to all

# 15 — Failure testing
# FAIL-A (incomplete request): validator must demand details, invent nothing.
# FAIL-B (simulated sky timeout): mock requests.get to raise Timeout, prove
# SkyWatch returns unavailable, HazardRadar degrades, crew survives.


In [24]:
# FAIL-A: incomplete request (real validator, no crew needed)
fa = "Organise a college event."
print("FAIL-A INPUT:", fa)
va = check_input(fa)
print("validator:", json.dumps(va, indent=2))
print("brief tool:", p_tool._run(fa)[:400])
print("provisional:", run_campus_assistant(fa)["final"][:300] if (NEXUSAI_OK and ag1 is not None) else "crew skipped (no key); validator + brief above already demand headcount/duration/venue.")
print("PASS if: missing items listed, zero invented dates/venues/guests.")


FAIL-A INPUT: Organise a college event.
validator: {
  "status": "ok",
  "missing": [
    "headcount",
    "duration",
    "venue"
  ],
  "questions": [
    "Please confirm headcount?",
    "Please confirm duration?",
    "Please confirm venue?"
  ]
}
brief tool: {
  "facts_from_user": {
    "event_type": "campus event"
  },
  "missing": [
    "headcount",
    "duration_hours",
    "venue"
  ],
  "questions": [
    "Please confirm headcount?",
    "Please confirm duration_hours?",
    "Please confirm venue?"
  ],
  "rule": "Unlisted items are NOT confirmed."
}
provisional: Workflow stopped safely. Diagnostic: RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.
PASS if: missing items listed, zero invented dates/venues/guests.


In [25]:
# FAIL-B: SIMULATED SkyWatch timeout via mock (labelled simulation)
from unittest.mock import patch
print("FAIL-B: forcing requests.get to raise Timeout...")
with patch("requests.get", side_effect=requests.exceptions.Timeout("simulated timeout")):
    sim = w_tool._run(12.97, 74.88, 2)
print("SkyWatch says:", sim[:300])
print("hazards on unavailable:", r_tool._run(True, 200, sim)[:500])
ok = '"unavailable"' in sim and "timeout" in sim
print("PASS" if ok else "REVIEW", ": unavailable flagged, no forecast invented, pipeline continues.")


FAIL-B: forcing requests.get to raise Timeout...
SkyWatch says: {"status": "unavailable", "reason": "timeout", "guidance": "Proceed with indoor backup."}
hazards on unavailable: {
  "sky_used": "unavailable (assumptions flagged)",
  "risks": [
    {
      "risk": "unknown sky for outdoor segment",
      "level": "MEDIUM",
      "fix": "book indoor backup; recheck sky 24h before"
    },
    {
      "risk": "power/Wi-Fi drop mid-hacking",
      "level": "MEDIUM",
      "fix": "UPS per row, backup router + offline judging sheets"
    },
    {
      "risk": "judging overrun",
      "level": "LOW",
      "fix": "3-min demo timer + overflow room"
    }
  ]
}
PASS : unavailable flagged, no forecast invented, pipeline continues.


# 16 — Debug log (fill from YOUR runs; starters below are real local findings)
| # | Problem | Behaviour | Cause | Fix | Leftover |
|---|---|---|---|---|---|
| 1 | Pydantic v2 rejects `args_schema` typing | Tool class fails at import | Schema must be `type[BaseModel]` | Used `args_schema: type[BaseModel]` in all 8 tools | None |
| 2 | CrewAI BaseTool import path differs by version | ImportError on older installs | Moved between `crewai.tools` and `crewai_tools` | try/except import in cell 05 | None |
| 3 | SkyWatch flaky on slow networks | Timeout exception | No timeout set | 15 s timeout + unavailable JSON (tested in FAIL-B) | Recheck sky 24 h before event |
| 4 | (your run) | | | | |


# 17 — Results and observations (write what YOU see)
# Tool demos: all 8 fire in cell 21 (check CALL_LOG length = 8+).
# Sky probe: live values in cell 11, or unavailable with reason.
# Crew: needs NEXUSAI_API_KEY; compare final vs tool math before submitting.
# Honest limits: equal-slot timetable ignores session weights; costs are
# estimates; sky is Mangalore-grid not your exact hall; Nexus drafts need proofread.


# 18 — Conclusion
# One request flows through Design -> Ops/Finance -> Safety -> Outreach.
# Python owns the numbers, Open-Meteo owns the sky, Nexus owns the words.
# Unknowns stay [TBD]; failures degrade instead of crashing. To extend:
# add a judging-score tool, a sponsor-mail tool, or a room-allotment checker —
# each as a new BaseTool with its own agent task context.


# 19 — Instructions for running the notebook
# 1. Open in Google Colab (badge above).
# 2. Secrets panel: add NEXUSAI_API_KEY (required), NEXUSAI_BASE_URL + NEXUSAI_MODEL_ID (optional).
# 3. Runtime -> Run all, top to bottom. Cells 04-11 prove setup; 13-21 prove tools.
# 4. Edit HACK_REQUEST (cell 27) for your own hackathon; re-run 30-37.
# 5. Submit the .ipynb with outputs kept. Never paste keys into code or outputs.
# File: VALLUR SREE VARDHAN_NNM24CS285_Assignment2.ipynb | Student: VALLUR SREE VARDHAN (NNM24CS285)
